# Capital Adequacy Ratio (CAR) - Data Cleaning 

### Data Info:
* **Indicator:** `Норматив достатності (адекватності) регулятивного капіталу (не менше 10 %) (CAR)`
* **Source:** National Bank of Ukraine: https://bank.gov.ua/ua/statistic/supervision-statist
* **Raw File:** `data_raw/CAR.xlsx`
* **Output:** `data_processed/CAR_clean.xlsx`  

### Cleaning Notes:
- `…` in the values row (-> we replace it with NaN)

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

In [2]:
file_path = Path("../../data_raw/CAR.xlsx")
output_path = Path("../../data_processed/CAR_clean.xlsx")
output_path.parent.mkdir(parents=True, exist_ok=True)

xls = pd.ExcelFile(file_path)
sheet_names = xls.sheet_names

print(sheet_names)

['2025', '2024', '2023', '2022', '2021', '2020', '2019', '2018', '2017', '2016', '2015']


In [3]:
# inspect
for sheet in sheet_names[:11]:
    df_preview = pd.read_excel(file_path, sheet_name=sheet, header=None)

    print(f"\n--- sheet: {sheet} ---")
    print("dates row:")
    print(df_preview.iloc[3, 3:15].tolist())

    print("values row:")
    print(df_preview.iloc[5, 3:15].tolist())


--- sheet: 2025 ---
dates row:
['01/01', '01/02', '01/03', '01/04', '01/05', '01/06', '01/07', '01/08', '01/09', '01/10', '01/11', '01/12']
values row:
[17.35, 16.98, 16.91, 16.23, 15.49, 15.49, 15.52, 15.31, 15.02, 15.43, 15.26, 15.08]

--- sheet: 2024 ---
dates row:
['01/01', '01/02', '01/03', '01/04', '01/05', '01/06', '01/07', '01/08', '01/09', '01/10', '01/11', '01/12']
values row:
[21.07, 20.57, 19.98, 20.44, 18.38, 19.07, 19.07, 19.85, '…', '…', '…', '…']

--- sheet: 2023 ---
dates row:
['01/01', '01/02', '01/03', '01/04', '01/05', '01/06', '01/07', '01/08', '01/09', '01/10', '01/11', '01/12']
values row:
[19.68, 19.84, 20.34, 20.8, 20.95, 22.8, 23.77, 24.3, 24.94, 24.99, 25.31, 25.41]

--- sheet: 2022 ---
dates row:
['01/01', '01/02', '01/03', '01/04', '01/05', '01/06', '01/07', '01/08', '01/09', '01/10', '01/11', '01/12']
values row:
[18.01, 17.99, '…', '…', '…', 16.65, 17.16, 17.1, 16.92, 18.85, 18.84, 19.16]

--- sheet: 2021 ---
dates row:
['01/01', '01/02', '01/03', '01/04

In [4]:
# set correct header

all_parts = []

for sheet in sheet_names:
    
    df_raw = pd.read_excel(file_path, sheet_name=sheet, header=None)
    
    date_cells = df_raw.iloc[3, 3:]
    value_cells = df_raw.iloc[5, 3:]
    indicator_name = "CAR"
    
    part = pd.DataFrame({
        "date_raw": date_cells.values,
        "value": value_cells.values
    })
    part["sheet_year"] = str(sheet)
    part["indicator"] = indicator_name
    
    all_parts.append(part)

df = pd.concat(all_parts, ignore_index=True)
df

,date_raw,value,sheet_year,indicator
0,01/01,17.35,2025,CAR
1,01/02,16.98,2025,CAR
2,01/03,16.91,2025,CAR
3,01/04,16.23,2025,CAR
4,01/05,15.49,2025,CAR
...,...,...,...,...
127,01/08,8.03,2015,CAR
128,01/09,7.88,2015,CAR
129,01/10,7.09,2015,CAR
130,01/11,9.9,2015,CAR


In [5]:
# parse dates

df["date"] = pd.to_datetime(
    df["date_raw"].astype(str).str.strip() + "/" + df["sheet_year"].astype(str),
    format="%d/%m/%Y",
    errors="coerce"
)

print(df[["date_raw", "sheet_year", "date"]].head(15))

   date_raw sheet_year       date
0     01/01       2025 2025-01-01
1     01/02       2025 2025-02-01
2     01/03       2025 2025-03-01
3     01/04       2025 2025-04-01
4     01/05       2025 2025-05-01
5     01/06       2025 2025-06-01
6     01/07       2025 2025-07-01
7     01/08       2025 2025-08-01
8     01/09       2025 2025-09-01
9     01/10       2025 2025-10-01
10    01/11       2025 2025-11-01
11    01/12       2025 2025-12-01
12    01/01       2024 2024-01-01
13    01/02       2024 2024-02-01
14    01/03       2024 2024-03-01


In [6]:
# clean

def clean_value(x):

    if pd.isna(x):
        return np.nan

    s = str(x).strip()

    if s in ["...", "…"]:
        return np.nan 

    s = s.replace(",", ".")

    try:
        return float(s)
    except:
        return np.nan

df["value"] = df["value"].apply(clean_value)
df[["date", "value"]]

,date,value
0,2025-01-01,17.35
1,2025-02-01,16.98
2,2025-03-01,16.91
3,2025-04-01,16.23
4,2025-05-01,15.49
...,...,...
127,2015-08-01,8.03
128,2015-09-01,7.88
129,2015-10-01,7.09
130,2015-11-01,9.90


In [7]:
# standardize indicator

df_final = df[["date", "value", "indicator"]].copy()
df_final = df_final.sort_values("date").reset_index(drop=True)

df_final.head(15)

,date,value,indicator
0,2015-01-01,15.60,CAR
1,2015-02-01,13.81,CAR
2,2015-03-01,7.37,CAR
3,2015-04-01,8.35,CAR
4,2015-05-01,7.84,CAR
5,2015-06-01,7.66,CAR
6,2015-07-01,9.03,CAR
7,2015-08-01,8.03,CAR
8,2015-09-01,7.88,CAR
9,2015-10-01,7.09,CAR


In [8]:
# validate

print(df_final.dtypes)
print(df_final["date"].isna().sum())

year_count = (
    df_final
    .assign(year=df_final["date"].dt.year)
    .groupby("year")
    .size()
)

print(year_count)

date         datetime64[ns]
value               float64
indicator            object
dtype: object
0
year
2015    12
2016    12
2017    12
2018    12
2019    12
2020    12
2021    12
2022    12
2023    12
2024    12
2025    12
dtype: int64


In [9]:
# save

df_final["date"] = df_final["date"].dt.date
df_final.to_excel(output_path, index=False)
print(f"saved to: {output_path}")

saved to: ..\..\data_processed\CAR_clean.xlsx
